In [7]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

import sys
import os
import numpy as np
import pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/results"
if RESULTS_DIR not in sys.path:
    sys.path.append(RESULTS_DIR)

model_summary = os.path.join(RESULTS_DIR, "summary_clean.csv")

if os.path.exists(model_summary):
  print("Model summary exists")
  df = pd.read_csv(model_summary)
else:
  raise Exception("Model summary does not exist")

agg = df.groupby("model").agg({
    "f1_mean": "mean",
    "accuracy_mean": "mean",
    "f1_std": "mean"
}).reset_index()

def normalize(col):
  col = col.fillna(col.mean())
  return (col - col.min()) / (col.max() - col.min() + 1e-8)

agg["f1_norm"] = normalize(agg["f1_mean"])
agg["acc_norm"] = normalize(agg["accuracy_mean"])
agg["std_norm"] = normalize(agg["f1_std"])

w1, w2, w3 = 0.6, 0.3, 0.1

agg["score"] = (
    w1 * agg["f1_norm"] +
    w2 * agg["acc_norm"] -
    w3 * agg["std_norm"]
)

agg = agg.sort_values(by="score", ascending=False)
agg["rank"] = np.arange(1, len(agg) + 1)

table = agg[[
    "rank",
    "model",
    "f1_mean",
    "accuracy_mean",
    "f1_std",
    "score"
]].copy()

table = table.round({
    "f1_mean": 3,
    "accuracy_mean": 3,
    "f1_std": 3,
    "score": 3
})

print(table)
print(table.to_latex(index=False))

table.to_csv(os.path.join(RESULTS_DIR, "model_overall_ranking.csv"), index=False)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model summary exists
   rank       model  f1_mean  accuracy_mean  f1_std  score
6     1   tinyllama    0.743          0.837   0.374  0.788
4     2          rf    0.603          0.836   0.000  0.755
1     3  distilbert    0.594          0.868   0.373  0.659
0     4     cnnlstm    0.448          0.776   0.016  0.579
2     5         mae    0.338          0.579   0.113  0.374
5     6       stmae    0.112          0.717   0.068  0.224
3     7         mlp    0.195          0.089   0.008  0.077
\begin{tabular}{rlrrrr}
\toprule
rank & model & f1_mean & accuracy_mean & f1_std & score \\
\midrule
1 & tinyllama & 0.743000 & 0.837000 & 0.374000 & 0.788000 \\
2 & rf & 0.603000 & 0.836000 & 0.000000 & 0.755000 \\
3 & distilbert & 0.594000 & 0.868000 & 0.373000 & 0.659000 \\
4 & cnnlstm & 0.448000 & 0.776000 & 0.016000 & 0.579000 \\
5 & mae & 0.338000 & 0.579000 & 0.113000 